*   Image classification uses images as inputs and class IDs as labels.
*   This notebook uses tiny synthetic image tensors so the data shape and batching ideas are visible without downloading a dataset.

In [ ]:
import math
import random
import numpy as np
import torch

# 4.2.1 Loading the Dataset

## 1. Intuition

*   An image dataset contains image tensors and labels.
> *   A grayscale image can be stored as a matrix of pixel values.
> *   A pixel is one small measurement in an image.
> *   A channel is one layer of pixel measurements, such as grayscale, red, green, or blue.

## 2. Why this exists

*   Image models need a consistent tensor shape.
*   For many PyTorch image datasets, the shape is `(examples, channels, height, width)`.
> * In this example, `softmax` should be applied over (`dim=1`) because channels represent classes.
> * Do not assume `channels` equals `classes`; check what each dimension represents before choosing `dim`.

## 3. Examples

*   Create a tiny image dataset with four grayscale images.

In [ ]:
images = torch.arange(4 * 1 * 2 * 2, dtype=torch.float32)
images = images.reshape(4, 1, 2, 2) # (# of images, channels, height, width); Each image is a 2×2 grayscale image.

labels = torch.tensor([0, 1, 0, 1]) # Classify images into classes of 0 and 1; With images[0] and images[2] under label 0 (e.g. both cat pictures) and images[1] and images[3] under label 1 (e.g. both dog pictures)

# Combines(images, labels) into datasets; dataset[0] -> (images[0], labels[0])....
# Length is 4 because TensorDataset treats the 1st dimension of the tensors as the number of examples
dataset = torch.utils.data.TensorDataset(images, labels)

len(dataset), images.shape, labels.shape # 4; (4, 1, 2, 2); (4)

(4, torch.Size([4, 1, 2, 2]), torch.Size([4]))

## 4. Step-by-step breakdown

*   `torch.arange(...)` creates enough pixel values for four small images.

*   `reshape(4, 1, 2, 2)` means 4 examples, 1 channel, height 2, width 2.

*   `labels` stores one class ID per image.

*   `TensorDataset(images, labels)` keeps each image aligned with its label.

## 5. Connection to ML systems

*   Real datasets such as Fashion-MNIST follow the same idea but with many more images and larger spatial dimensions.

## 6. Common confusion points

- Image tensors have shape conventions; always check them.
- [**Labels should align with the first dimension of images**].
- Pixel values are numbers, even when the image looks visual to humans.
- Loading data is separate from training a model.

# 4.2.2 Reading a Minibatch

## 1. Intuition

*   A minibatch is a small group of examples returned together for one training step.

*   For images, a minibatch keeps the image dimensions and adds or preserves the batch dimension.

## 2. Why this exists

*   Training one image at a time wastes hardware efficiency.
*   Training all images at once may use too much memory.
*   Minibatches balance both concerns.

## 3. Examples

*   Read one minibatch using a DataLoader.

In [ ]:
loader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True) # Breaks down our dataset into minibatches of size 2, or 2 batches (2*2=4)
batch_images, batch_labels = next(iter(loader)) # Spit out each bath iteratively
batch_images.shape, batch_labels.shape # (2, 1, 2, 2) (2)

(torch.Size([2, 1, 2, 2]), torch.Size([2]))

*   Flatten images when using a linear classifier.
> * Flattening is done to convert the image into a 1D feature vector so that a fully connected linear layer can process it.
> * In this instance, instead of `(# of images, channels, width, height)`, the data becomes (# of iamges, # of features).
> * For a 2×2 grayscale image: 1 * 2 * 2 = 4 features, so `(2, 1, 2, 2)` -> `(2, 4)`
* Flattening does not create probabilities; it only changes the representation of the image.
* After flattening, a linear classifier can process the features and produce logits. During training, the model typically uses:
> * `CrossEntropyLoss()`,
> * `loss.backward()`, finally
> * `optimizer.step()` to update its weights

In [ ]:
flat = batch_images.reshape(batch_images.shape[0], -1) # The 1st dimension is 2 and stay as, the remainder dimensions are recalculated so that the total number of elements conserve, or 1 * 2 * 2 = 4
flat.shape # (2, 4)

torch.Size([2, 4])

## 4. Step-by-step breakdown

*   `DataLoader` groups dataset items into batches.

*   `next(iter(loader))` asks for the first batch.

*   `batch_images.shape` is `(2, 1, 2, 2)` for two tiny grayscale images.

*   `reshape(batch_images.shape[0], -1)` keeps the batch size and flattens each image into one feature vector.

## 5. Connection to ML systems

*   Softmax regression is a linear classifier, so it expects feature vectors.
*   For images, that often means flattening pixels before the linear layer.




## 6. Common confusion points

- Flattening removes spatial layout information.
- `-1` asks PyTorch to infer the remaining dimension.
- The batch dimension should stay first.
- Shuffling should not break image-label pairing.
> * When you enable `shuffle=True` (for example, in a PyTorch DataLoader), it shuffles the order of the pairs, not the images and labels independently.

# 4.2.3 Visualization

## 1. Intuition

*   Visualization means displaying data so humans can inspect it.

*   For images, visual inspection helps catch wrong labels, wrong shapes, inverted colors, or normalization mistakes.

## 2. Why this exists

*   Models only see numbers, but humans often need visual checks to verify the data pipeline.

## 3. Examples

*   Prepare one tiny image for display as a 2D grid.

In [6]:
image = images[0]
grid = image.squeeze(0) # Inspect dim=0 and removes it if it's 1. In this case, the channel dimension is 1 and removed, rendering our image to (2, 2) vs the original shae pf (1, 2, 2))
label = labels[0]
grid, label # grid: the 2×2 pixel values of the image; label: the correct class for that image.

(tensor([[0., 1.],
         [2., 3.]]),
 tensor(0))

*   A class-name lookup turns label IDs into readable names.

In [8]:
class_names = ["dark", "bright"]
name = class_names[int(label)]
name # dark, since label = labels[0], and class_names[0] is "dark"

'dark'

## 4. Step-by-step breakdown

*   `images[0]` selects the first image.
*   `squeeze(0)` removes the channel dimension when it has size 1.
*   The label is an integer class ID.
*   `class_names[int(label)]` maps the class ID to a human-readable name.

## 5. Connection to ML systems

*   Dataset visualization is usually done before training.
*   It is a cheap way to find data mistakes that no optimizer can fix.

## 6. Common confusion points

- Visualization is for inspection, not training.
- A label ID needs a class-name mapping to be readable.
- Squeezing removes dimensions only when their size is 1. [**`squeeze(dim)` only removes the specified dimension if its size is 1**].
- Do not confuse display shape with model input shape.

## `torch.squeeze()` examples

`squeeze(dim)` only removes the specified dimension **if its size is 1**.

#### Example 1: Remove the first dimension

```python
x.shape = (1, 3, 3)
x.squeeze(0).shape
# (3, 3)
```

Dimension `0` has size `1`, so it is removed.

---

#### Example 2: Nothing happens

```python
x.shape = (3, 3, 1)
x.squeeze(0).shape
# (3, 3, 1)
```

Nothing happens because dimension `0` has size `3`, not `1`.

---

#### Example 3: Remove the last dimension

```python
x.shape = (3, 3, 1)
x.squeeze(2).shape
# (3, 3)
```

Dimension `2` has size `1`, so it is removed.

---

#### Example 4: Remove all dimensions of size 1

If you omit the dimension, `squeeze()` removes **every** dimension whose size is `1`.

```python
x = torch.randn(1, 3, 1, 3, 1)
x.shape
# (1, 3, 1, 3, 1)

x.squeeze().shape
# (3, 3)
```

---

### Summary

- `squeeze()` → Removes **all** dimensions of size `1`.
- `squeeze(dim)` → Removes **only** the specified dimension, and **only if its size is `1`**.
- If the specified dimension's size is not `1`, the tensor is returned unchanged.

## `squeeze()` vs. `flatten()`

These are **two different operations** that are often discussed together, but they serve different purposes.

1. **`squeeze()`** removes dimensions **only if their size is `1`**.
2. **`flatten()`** reshapes an image into a **1D vector of features**.

They are **not** the same thing.

---

## For a CNN (Convolutional Neural Network)

**Do not squeeze away the channel dimension.**

A CNN expects input in the form:

```python
[batch_size, channels, height, width]
# e.g. [64, 1, 28, 28]
```

The channel dimension is important because convolutional layers operate on channels.

---

## For a Linear (Fully Connected) Model

A linear layer expects a **vector of features**, not a 2D image.

Starting with:

```python
[64, 1, 28, 28]
```

you typically flatten the images:

```python
images = images.view(images.size(0), -1)
```

or

```python
images = torch.flatten(images, start_dim=1)
```

which produces:

```python
[64, 784]
```

Notice that we **didn't need to call `squeeze()`**.

Flattening automatically combines the `1 × 28 × 28` dimensions into `784` features while preserving the batch dimension.

---

## Why use `squeeze()`?

`squeeze()` is commonly used for **displaying** grayscale images.

```python
image = images[0]          # Shape: [1, 28, 28]
grid = image.squeeze(0)    # Shape: [28, 28]

plt.imshow(grid, cmap="gray")
```

Matplotlib expects a **2D array** for grayscale images, so removing the channel dimension makes visualization easier.

---

## Summary

| Task | Shape |
|------|-------|
| Load a batch of images | `[64, 1, 28, 28]` |
| Display one grayscale image | `[28, 28]` using `squeeze(0)` |
| Feed images to a CNN | `[64, 1, 28, 28]` (keep the channel dimension) |
| Feed images to a Linear layer | `[64, 784]` using `flatten()` or `view()` |
```

# 4.2.4 Summary

## 1. Intuition

*   Image classification datasets pair image tensors with class labels.

*   The key shape convention is usually `(batch, channels, height, width)`.

## 2. Why this exists

*   Correct data shape and label alignment are prerequisites for meaningful classification training.

## 3. Examples

*   Summarize one batch's structure.

In [ ]:
summary = {
    "batch_size": batch_images.shape[0],
    "channels": batch_images.shape[1],
    "height": batch_images.shape[2],
    "width": batch_images.shape[3],
}

## 4. Step-by-step breakdown

*   The summary names each dimension.

*   Naming dimensions reduces confusion when tensors become larger.

*   The label tensor stores one class ID per image in the batch.

## 5. Connection to ML systems

*   Future image models will preserve spatial layout longer, but softmax regression starts by flattening images into feature vectors.

## 6. Common confusion points

- Always inspect batch shapes.
- Always preserve image-label alignment.
- Flattening is convenient but loses spatial structure.
- Class labels are usually integer IDs in PyTorch classification.

# 4.2.5 Exercises

## 1. Intuition

*   These exercises practice reading and reshaping image batches.

## 2. Why this exists

*   Most image-classification bugs start as data-shape bugs.

## 3. Examples

*   Exercise 1: create a batch of three 4 by 4 grayscale images.
*   [**Notice how different than regression models, `crossentropy` does not expect you to reshape `y`**].

    | Task           | Model output       | Target shape | Why                              |
    | -------------- | ------------------ | ------------ | -------------------------------- |
    | Classification | `(batch, classes)` | `(batch,)`   | Labels are class IDs (indices)   |
    | Regression     | `(batch, 1)`       | `(batch, 1)` | Labels are values being compared |

In [9]:
X = torch.zeros(3, 1, 4, 4)
y = torch.tensor([0, 1, 2])
X.shape, y.shape # (3, 1, 4, 4); (3) -> shape (1,3) would have an extra dimension like [[0, 1, 2]], not just [0, 1, 2]

(torch.Size([3, 1, 4, 4]), torch.Size([3]))

*   Exercise 2: flatten the image batch for a linear classifier.

In [10]:
flat = X.reshape(X.shape[0], -1)
flat.shape # (3, 1*4*4=16) or (3, 16)

torch.Size([3, 16])

## 4. Step-by-step breakdown

*   Exercise 1 checks the image batch shape convention.

*   Exercise 2 checks flattening while preserving the batch dimension.

*   The flattened feature count is `channels * height * width`.

## 5. Connection to ML systems

*   Flattened image batches feed directly into softmax regression and multilayer perceptrons.

## 6. Common confusion points

- The batch dimension is not an image dimension.
- Flatten only the dimensions that belong to each example.
- The number of labels should equal the batch size.
- The class ID range should match the number of classes.